# 23N Field Tests #10 · Gujarat's reservoirs, measured from space

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/23nlabs/field-tests/blob/ft10-reservoirs-from-space/ft10-reservoirs-from-space/ft10_reservoirs_from_space.ipynb)

**Question.** How full are Sardar Sarovar and three other major Gujarat reservoirs (Ukai, Kadana, Dharoi) each
month, measured from the area of open water in satellite images? And does that satellite measure move with the
storage levels the Central Water Commission (CWC) publishes?

**Why it matters.** CWC's weekly bulletin tracks 17 reservoirs in Gujarat with 17.964 BCM of live capacity
(bulletin of 10 Oct 2024). Those figures come from gauge readings at each dam and are published as weekly PDF
tables. A satellite gives an independent, repeatable check of the same reservoirs, from data anyone can download,
including for the many smaller reservoirs outside the bulletin.

**Method.** Copernicus Sentinel-2 Level-2A images (Microsoft Planetary Computer, no login). For each reservoir and
each month of 2022–2025 we take the clearest date, compute the Modified Normalised Difference Water Index
(MNDWI = (green − SWIR) / (green + SWIR), bands B03 and B11) on a 60 m grid, mask clouds with the Scene
Classification Layer (SCL), and count water pixels inside the reservoir's own zone (the connected water body at the
dam, found from the run's images). Months where clouds cover more than 5% of the zone are dropped. The satellite
"fill index" is the month's water area as a % of the largest area seen in the run. The notebook then downloads CWC
weekly bulletins (PDF), reads each reservoir's live storage, and compares the two on dates within 12 days.

**Data and licences.** Sentinel-2 L2A (collection `sentinel-2-l2a`): Copernicus Sentinel data terms (free, full
and open, commercial use allowed; attribution "Contains modified Copernicus Sentinel data 2022–2025"). **No account
and no key**: the `planetary-computer` package fetches a free, anonymous access token. CWC reservoir bulletins:
public Government of India PDFs from cwc.gov.in, used as published facts. No AI model weights are used: the water
map is a spectral index with a fixed threshold, written in the notebook.

**Runtime.** Roughly 15–25 minutes on free Colab or Kaggle CPU, almost all of it reading satellite images (only a
60 m overview of the window around each reservoir is read). `TIME_BUDGET_MIN` caps it.

*Independent study on public data by 23N AI Labs. The reservoirs are used only as places: this says nothing about
any dam authority, water board or ministry, and implies no affiliation with any of them. Water area is not
storage volume; every number is an estimate.*

## 1 · Setup

In [ ]:
# CELL: deps
# pystac-client finds the images, planetary-computer signs the links (anonymous, free), odc-stac reads just the
# window and resolution we need, pypdf reads the CWC bulletins.
import os, sys, subprocess
if os.environ.get("FT10_SMOKE_TEST", "0") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pystac-client", "planetary-computer", "odc-stac",
                    "pyproj", "shapely", "pypdf", "requests", "gradio"], check=False)

In [ ]:
# CELL: config
import os, sys, re, io, math, time, pickle, textwrap, subprocess, warnings, urllib.request, datetime as dt
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle/working")

# ---- Test switch. OFF by default. Only for build checks without internet: it replaces the satellite images and
# the CWC bulletins with synthetic stand-ins. Never post its output.
SMOKE_TEST = os.environ.get("FT10_SMOKE_TEST", "0") == "1"

FT_NUM = "10"
DATA_START, DATA_END = "2022-01-01", "2025-12-31"
END_YEAR = 2025                          # headline cycle: post-monsoon peak of END_YEAR-1 -> pre-monsoon low of END_YEAR
PEAK_MONTHS = [9, 10, 11, 12]           # reservoirs are fullest after the south-west monsoon
LOW_MONTHS = [3, 4, 5, 6]               # and lowest before the next one
MONSOON_MONTHS = [6, 7, 8, 9]

# ---- The reservoirs. ASSUMPTION: each is searched for inside a lon/lat window (lon_min, lat_min, lon_max, lat_max)
# drawn generously around the reservoir; the reservoir itself is the connected water body within SEED_RADIUS_KM of a
# published point. Points: Sardar Sarovar dam 21.8303 N, 73.7472 E (Wikipedia); Ukai reservoir 21.3235 N,
# 73.7073 E (DAHITI, TU Munich, id 238); Kadana dam 23.3073 N, 73.8273 E (Wikipedia); Dharoi dam 24.0044 N,
# 72.8536 E (Wikipedia). CWC names are the names printed in the CWC weekly bulletin.
RESERVOIRS = {
    "Sardar Sarovar": dict(seed=(21.8303, 73.7472), box=(73.70, 21.70, 75.20, 22.35), cwc="SARDAR SAROVAR",
                           live_bcm=5.760, area_km2=375.0, full_dates=["2023-09-17", "2025-10-01"]),
    "Ukai":           dict(seed=(21.3235, 73.7073), box=(73.50, 21.10, 74.50, 21.60), cwc="UKAI",
                           live_bcm=6.615, area_km2=520.0, full_dates=["2024-10-10"]),
    "Kadana":         dict(seed=(23.3073, 73.8273), box=(73.75, 23.20, 74.25, 23.65), cwc="KADANA",
                           live_bcm=1.472, area_km2=None, full_dates=[]),
    "Dharoi":         dict(seed=(24.0044, 72.8536), box=(72.75, 23.95, 73.10, 24.25), cwc="DHAROI",
                           live_bcm=0.735, area_km2=107.0, full_dates=[]),
}
RES_ORDER = list(RESERVOIRS)
# Documented facts used by the sanity checks (sources in the README):
#   area_km2: Sardar Sarovar 375.33 km2 surface area (Wikipedia; 37,000 ha per sardarsarovardam.org); Ukai water
#   spread "about 52,000 hectares"; Dharoi surface area 107 km2 (Wikipedia, level not stated). Kadana: none found.
#   live_bcm: live capacity at FRL in the CWC bulletin of 10 Oct 2024 (used to check the PDF parsing).
#   full_dates: Sardar Sarovar reached its full level of 138.68 m on 17 Sep 2023 and 1 Oct 2025 (DeshGujarat);
#   Ukai was at 100% of live capacity on 10 Oct 2024 (CWC bulletin).
SEED_RADIUS_KM = 5.0

# ---- Satellite data
STAC_URL = "https://planetarycomputer.microsoft.com/api/stac/v1"
COLLECTION = "sentinel-2-l2a"
BANDS = ["B03", "B11", "SCL"]
CRS = "EPSG:32643"                      # UTM zone 43N covers 72-78 E: every reservoir here, in metres
RES_M = 60                              # grid size; 60 m reads COG overviews and keeps the run on free CPU
PIXEL_KM2 = (RES_M / 1000) ** 2
ITEM_CLOUD_MAX = 60                     # skip tiles whose own cloud cover is above this (%)
TRIES_PER_MONTH = 2                     # dates tried per month, clearest first
BOX_CLOUD_MAX = 0.40                    # try the next date if more than this share of the window is cloud
MIN_BOX_COVER = 0.97                    # a date's tiles must cover this share of the window
MNDWI_THR = 0.0                         # water if MNDWI > 0 (the usual MNDWI threshold)
MNDWI_ROBUST = (-0.1, 0.1)              # robustness: thresholds either side
ZONE_MIN_FREQ = 0.05                    # a pixel belongs to the reservoir zone if water in >= 5% of its clear looks
ZONE_DILATE_PX = 2
ZONE_CLOUD_MAX = 0.05                   # drop a month if more than 5% of the zone is cloud or missing
BASELINE_CUTOFF = "2022-01-25"          # ESA processing baseline 04.00: +1000 offset added to reflectances
HARMONIZE_OFFSET = 1000
SCL_NODATA, SCL_CLOUD, SCL_WATER = [0, 1], [3, 8, 9, 10], 6
TIME_BUDGET_MIN = 15
WORKERS = 6

# ---- CWC weekly bulletins (PDF). Every URL below was found on cwc.gov.in (Nov 2026). CWC's file names vary, so
# CWC_PROBE also tries the naming patterns seen in that list for one Thursday per month that has no listed bulletin;
# a probed file is used only if it downloads, parses and its live capacities match.
CWC_BASE = "https://cwc.gov.in/sites/default/files/"
CWC_URLS = [CWC_BASE + f for f in [
    "17032022-bulletin.pdf", "24032022-bulletin.pdf", "19052022-bulletin.pdf", "26052022.pdf", "2306-2022-bulletin.pdf",
    "01092022.pdf", "29092022-fb.pdf", "13102022-fb.pdf", "27102022-fb.pdf", "10112022-fb.pdf", "24112022-fbpdf.pdf",
    "01122022-fb.pdf", "29122022-fb.pdf", "230323-fb.pdf", "13042023-fb.pdf", "20042023-fb.pdf", "04052023-fb.pdf",
    "18052023-fb_1.pdf", "25052023-fb.pdf", "fb-13072023.pdf", "fb-03082023.pdf", "05102023-fb.pdf",
    "12042024-bulletin.pdf", "fb-20062024.pdf", "fb10102024.pdf", "fb-14112024_1.pdf", "fb-21112024.pdf",
    "fb-16012025.pdf", "23012025-fbpdf_1.pdf", "fb30012025.pdf", "fb-06022025.pdf", "fb-13022025.pdf",
    "20022025-fb-pdf.pdf", "06032025-fb-pdf.pdf", "20032025-fb.pdf", "fb-01052025.pdf", "bulletin-08-05-2025.pdf"]] + [
    "https://rsms.cwc.gov.in/admin/storage/bulletins/bulletin-18-12-2025-70.pdf"]
CWC_PROBE = True
CWC_PROBE_BUDGET_MIN = 3
CWC_PATTERNS = ["fb{d}.pdf", "fb-{d}.pdf", "{d}-fb.pdf", "{d}-fb-pdf.pdf", "{d}-bulletin.pdf", "bulletin-{dd}-{mm}-{yyyy}.pdf"]
CWC_CAPACITY_TOL = 0.10                 # a parsed row is kept only if its live capacity is within 10% of live_bcm
MATCH_DAYS = 12                         # bulletin and satellite image must be within this many days
CWC_FACT = ("CWC's weekly bulletin tracks 17 reservoirs in Gujarat, with 17.964 BCM of live capacity "
            "(bulletin of 10 Oct 2024)")

# ---- Sanity-check bounds (what is already known)
MIN_CLEAR_MONTHS = 18                   # of 48, per reservoir
AREA_RATIO_RANGE = (0.5, 1.2)           # largest measured area vs the documented surface area
FULL_DATE_MIN_FILL = 0.85               # on a documented full date, the fill index should be >= 85%
MAX_DRAWDOWN = 0.90                     # losing > 90% of the water surface in one dry season would be extraordinary
MIN_RHO = 0.5                           # satellite vs CWC rank correlation expected at least this
MIN_CWC_PAIRS = 6
SCL_AGREE_MAX = 0.35                    # median |MNDWI area - SCL water area| / MNDWI area

# ---- Where things live (this branch until it is merged)
REPO_REF = "ft10-reservoirs-from-space"
RAW = f"https://raw.githubusercontent.com/23nlabs/field-tests/{REPO_REF}"

# ---- Brand kit
if not os.path.exists("brand23n.py"):
    urllib.request.urlretrieve(f"{RAW}/brand/brand23n.py", "brand23n.py")
import brand23n as B
B.setup()
DATA_CREDIT = "Contains modified Copernicus Sentinel data 2022-2025, via Microsoft Planetary Computer; storage: CWC bulletins"
print("Colab:", IN_COLAB, "| Kaggle:", IN_KAGGLE, "| SMOKE_TEST:", SMOKE_TEST)
if SMOKE_TEST:
    print("!! SMOKE TEST MODE: synthetic images and synthetic bulletins. These results are NOT real. Do NOT post.")

## 2 · Data: Sentinel-2 over four reservoirs, 2022–2025, and CWC's weekly storage

The Planetary Computer STAC API lists every Sentinel-2 L2A tile. For each reservoir window and month we rank the
dates by their tiles' cloud cover, check that the date's tiles cover the whole window, and read bands B03 (green),
B11 (short-wave infrared) and SCL (scene classification) at 60 m. From 25 Jan 2022 ESA's processing baseline 04.00
adds +1000 to every reflectance value; we subtract it (the Planetary Computer's own recommendation), and print the
baselines found, because a missed offset would bias the water index.

In [ ]:
# CELL: loaders
from pyproj import Transformer
TO_UTM = Transformer.from_crs("EPSG:4326", CRS, always_xy=True)

def month_list():
    return list(pd.period_range(DATA_START, DATA_END, freq="M"))

def item_baseline(it):
    b = it.properties.get("s2:processing_baseline")
    try:
        return float(b)
    except Exception:
        return None

def needs_offset(items, day):
    bs = [item_baseline(i) for i in items]
    bs = [b for b in bs if b is not None]
    return (min(bs) >= 4.0) if bs else (pd.Timestamp(day) >= pd.Timestamp(BASELINE_CUTOFF))

def search_items(name):
    import pystac_client
    from shapely.geometry import shape, box as sbox
    from shapely.ops import unary_union
    bx = RESERVOIRS[name]["box"]; win = sbox(*bx)
    cat = pystac_client.Client.open(STAC_URL)
    items = []
    for year in range(int(DATA_START[:4]), int(DATA_END[:4]) + 1):
        rng = f"{year}-01-01/{year}-12-31"
        try:
            items += list(cat.search(collections=[COLLECTION], bbox=list(bx), datetime=rng,
                                     query={"eo:cloud_cover": {"lt": ITEM_CLOUD_MAX}}).items())
        except Exception as e:                                       # query extension refused: filter here
            print("  STAC query filter refused, filtering client-side:", type(e).__name__)
            items += [i for i in cat.search(collections=[COLLECTION], bbox=list(bx), datetime=rng).items()
                      if i.properties.get("eo:cloud_cover", 100) < ITEM_CLOUD_MAX]
    by_day = {}
    for it in items:
        by_day.setdefault(pd.Timestamp(it.datetime).tz_localize(None).normalize(), []).append(it)
    rows = []
    for day, its in by_day.items():
        cover = unary_union([shape(i.geometry) for i in its]).intersection(win).area / win.area
        if cover >= MIN_BOX_COVER:
            rows.append(dict(day=day, items=its, cloud=float(np.mean([i.properties.get("eo:cloud_cover", 100) for i in its])),
                             baseline=",".join(sorted({str(i.properties.get("s2:processing_baseline", "?")) for i in its}))))
    cand = pd.DataFrame(rows)
    print(f"  {name}: {len(items)} tiles, {len(cand)} dates covering the window")
    return cand

def read_scene(name, items, day):
    import odc.stac, planetary_computer as pc
    ds = odc.stac.load(items, bands=BANDS, bbox=RESERVOIRS[name]["box"], crs=CRS, resolution=RES_M,
                       groupby="solar_day", resampling={"B03": "average", "B11": "average", "SCL": "nearest"},
                       patch_url=pc.sign, fail_on_error=False, chunks=None)
    ds = ds.isel(time=0)
    g = ds["B03"].values.astype(np.float32); s = ds["B11"].values.astype(np.float32); scl = ds["SCL"].values
    off = HARMONIZE_OFFSET if needs_offset(items, day) else 0
    return encode(g, s, scl, off), ds["x"].values, ds["y"].values

def encode(g, s, scl, off):
    # int8 code per pixel: -128 no data, -127 cloud, else round(MNDWI x 100); plus SCL's own water flag
    valid = (g > 0) & (s > 0) & ~np.isin(scl, SCL_NODATA)
    g = np.clip(g - off, 1, None); s = np.clip(s - off, 1, None)
    mndwi = (g - s) / (g + s)
    code = np.where(valid, np.clip(np.round(mndwi * 100), -100, 100), -128).astype(np.int8)
    code[valid & np.isin(scl, SCL_CLOUD)] = -127
    return code, (scl == SCL_WATER) & valid

def load_real():
    SC, GRID, LOG = {n: {} for n in RES_ORDER}, {}, []
    cands = {n: search_items(n) for n in RES_ORDER}
    tasks = []                                                     # (order key, reservoir, month)
    for mi, m in enumerate(month_list()):
        for ri, n in enumerate(RES_ORDER):
            tasks.append(((mi % 3, mi, ri), n, m))
    tasks.sort()                                                   # every 3rd month first: a cut-off run stays spread out
    def work(n, m):
        c = cands[n]
        if c.empty:
            return n, m, None, "no dates"
        c = c[c.day.dt.to_period("M") == m].sort_values("cloud").head(TRIES_PER_MONTH)
        if c.empty:
            return n, m, None, "no usable date"
        for r in c.itertuples():
            try:
                (code, sclw), x, y = read_scene(n, r.items, r.day)
            except Exception as e:
                last = f"{type(e).__name__}: {str(e)[:100]}"; continue
            v = code != -128
            cloud = float(((code == -127) & v).sum() / max(v.sum(), 1))
            if v.mean() >= 0.9 and cloud <= BOX_CLOUD_MAX:
                return n, m, dict(day=r.day, code=code, sclw=sclw, x=x, y=y, baseline=r.baseline,
                                  offset=needs_offset(r.items, r.day)), "ok"
            last = f"cloud {cloud:.0%}, valid {v.mean():.0%}"
        return n, m, None, last
    t0, done = time.time(), 0
    with ThreadPoolExecutor(WORKERS) as ex:
        it = iter(tasks); running = {}
        def submit_next():
            try:
                _, n, m = next(it)
            except StopIteration:
                return
            running[ex.submit(work, n, m)] = (n, m)
        for _ in range(WORKERS * 2):
            submit_next()
        while running:
            f = next(as_completed(list(running))); n, m = running.pop(f)
            try:
                n, m, sc, msg = f.result()
            except Exception as e:
                sc, msg = None, f"{type(e).__name__}"
            if sc is not None:
                GRID.setdefault(n, (sc.pop("x"), sc.pop("y")))
                sc.pop("x", None); sc.pop("y", None)
                SC[n][str(m)] = sc
            LOG.append(dict(reservoir=n, month=str(m), status=msg)); done += 1
            if done % 20 == 0:
                el = time.time() - t0
                print(f"  {done}/{len(tasks)} months · {el / 60:.1f} min · {el / done:.1f} s each (parallel)")
            if (time.time() - t0) / 60 < TIME_BUDGET_MIN:
                submit_next()
    if done < len(tasks):
        print(f"TIME_BUDGET_MIN reached: {done} of {len(tasks)} reservoir-months read (spread across the period)")
    return SC, GRID, pd.DataFrame(LOG)

def synthetic_data(seed=10):
    # SMOKE TEST ONLY: each reservoir is a valley; the water line rises each monsoon and falls through the dry
    # season; random clouds, heavy in the monsoon. Not real data.
    rng = np.random.default_rng(seed)
    SC, GRID, LOG = {n: {} for n in RES_ORDER}, {}, []
    for k, n in enumerate(RES_ORDER):
        lo0, la0, lo1, la1 = RESERVOIRS[n]["box"]
        (x0, x1), (y0, y1) = TO_UTM.transform([lo0, lo1], [la0, la1])
        W = min(int((x1 - x0) / RES_M), 420); H = min(int((y1 - y0) / RES_M), 220)
        step_x, step_y = (x1 - x0) / W, (y1 - y0) / H
        x = x0 + step_x * (np.arange(W) + 0.5); y = y1 - step_y * (np.arange(H) + 0.5)
        sx, sy = TO_UTM.transform(RESERVOIRS[n]["seed"][1], RESERVOIRS[n]["seed"][0])
        X, Y = np.meshgrid(x, y)
        # a winding valley starting at the seed and running east
        cy = sy + 0.25 * (y1 - y0) * np.sin((X - sx) / (x1 - x0) * 5)
        dist = np.abs(Y - cy) / ((y1 - y0) * 0.5)
        elev = dist * 0.9 + np.clip((X - sx) / (x1 - x0), 0, 1) * 0.6 + rng.normal(0, 0.02, X.shape)
        elev[X < sx - 1500] = 2.0                                    # the dam
        GRID[n] = (x, y)
        for m in month_list():
            if rng.random() < (0.6 if m.month in MONSOON_MONTHS else 0.08):
                LOG.append(dict(reservoir=n, month=str(m), status="cloud 80%")); continue
            day = m.to_timestamp() + pd.Timedelta(days=int(rng.integers(0, 27)))
            ph = (day.dayofyear - 290) / 365.25 * 2 * np.pi
            fill = 0.55 + 0.4 * np.cos(ph) - 0.05 * k * (day.year == 2024)
            level = 0.15 + 0.6 * fill
            water = elev < level
            cloud = np.zeros_like(water)
            for _ in range(int(rng.integers(0, 3))):
                cx, cy2, r = rng.integers(0, W), rng.integers(0, H), rng.integers(5, 25)
                cloud |= (np.arange(W)[None, :] - cx) ** 2 + (np.arange(H)[:, None] - cy2) ** 2 < r ** 2
            g = np.where(water, 900, 1100) + rng.normal(0, 40, water.shape)
            s = np.where(water, 300, 2200) + rng.normal(0, 60, water.shape)
            scl = np.where(water, 6, 4); scl = np.where(cloud, 9, scl)
            off = HARMONIZE_OFFSET if day >= pd.Timestamp(BASELINE_CUTOFF) else 0
            code, sclw = encode(g + off, s + off, scl, off)
            SC[n][str(m)] = dict(day=day, code=code, sclw=sclw, baseline="05.11" if off else "03.01", offset=bool(off))
            LOG.append(dict(reservoir=n, month=str(m), status="ok"))
    return SC, GRID, pd.DataFrame(LOG)

In [ ]:
# CELL: load_data
CACHE = f"ft{FT_NUM}_scenes.pkl"
if SMOKE_TEST:
    SC, GRID, READ_LOG = synthetic_data()
elif os.path.exists(CACHE):
    SC, GRID, READ_LOG = pickle.load(open(CACHE, "rb"))
    print("Loaded scenes from the local cache (delete", CACHE, "to download again).")
else:
    SC, GRID, READ_LOG = load_real()
    pickle.dump((SC, GRID, READ_LOG), open(CACHE, "wb"))

for n in RES_ORDER:
    shp = {v["code"].shape for v in SC[n].values()}
    print(f"{n:15s} {len(SC[n])} months read · grid {sorted(shp)} at {RES_M} m")
    if len(shp) > 1:                                                # guard: crop every scene to the common grid
        h, w = min(s[0] for s in shp), min(s[1] for s in shp)
        for v in SC[n].values():
            v["code"], v["sclw"] = v["code"][:h, :w], v["sclw"][:h, :w]
        GRID[n] = (GRID[n][0][:w], GRID[n][1][:h])
BASELINES = pd.DataFrame([dict(reservoir=n, year=v["day"].year, baseline=v["baseline"], offset=v["offset"])
                          for n in RES_ORDER for v in SC[n].values()])
if len(BASELINES):
    print("Processing baselines by year (from 04.00 the +1000 offset is removed):")
    print(BASELINES.groupby(["year", "baseline", "offset"]).size().rename("scenes").reset_index().to_string(index=False))
print("Months not used, by reason:")
print(READ_LOG[READ_LOG.status != "ok"].groupby("status").size().sort_values(ascending=False).head(8).to_string())

In [ ]:
# CELL: cwc_bulletins
def cwc_parse_text(text, src):
    # One row per reservoir: the numbers printed after its name, up to the row's date (dd-mm-yy). The live capacity
    # is found by value (within CWC_CAPACITY_TOL of live_bcm), and the live storage is the next number.
    rows = []
    flat = re.sub(r"[ \t\r\n]+", " ", text.upper())
    for n in RES_ORDER:
        r = RESERVOIRS[n]
        for mt in re.finditer(re.escape(r["cwc"]) + r"\)?[^0-9]{0,40}((?:\d+(?:\.\d+)? ){2,6})(\d{2}[-./]\d{2}[-./]\d{2,4})", flat):
            nums = [float(v) for v in mt.group(1).split()]
            idx = [i for i, v in enumerate(nums) if abs(v / r["live_bcm"] - 1) <= CWC_CAPACITY_TOL]
            if not idx or idx[0] + 1 >= len(nums):
                continue
            cap, sto = nums[idx[0]], nums[idx[0] + 1]
            if not (0 <= sto <= cap * 1.02):
                continue
            d = pd.to_datetime(mt.group(2).replace(".", "-").replace("/", "-"), dayfirst=True, errors="coerce")
            if pd.isna(d):
                continue
            rows.append(dict(reservoir=n, date=d.normalize(), live_bcm=sto, capacity_bcm=cap, pct=100 * sto / cap, source=src))
            break
    return rows

def cwc_fetch(url, timeout=25):
    import requests
    r = requests.get(url, timeout=timeout, headers={"User-Agent": "Mozilla/5.0 (23N Field Test notebook)"})
    if r.status_code != 200 or not r.content[:5] == b"%PDF-":
        return None
    from pypdf import PdfReader
    return "\n".join((p.extract_text() or "") for p in PdfReader(io.BytesIO(r.content)).pages)

def cwc_load_real():
    rows, status = [], []
    def one(url):
        try:
            t = cwc_fetch(url)
            return url, (cwc_parse_text(t, url) if t else None)
        except Exception as e:
            return url, f"{type(e).__name__}"
    with ThreadPoolExecutor(6) as ex:
        for url, res in ex.map(one, CWC_URLS):
            status.append((url, "ok" if isinstance(res, list) and res else "failed" if not isinstance(res, list) else "no rows"))
            if isinstance(res, list):
                rows += res
    if CWC_PROBE:
        import requests
        have = {pd.Timestamp(r["date"]).to_period("M") for r in rows}
        t0 = time.time()
        for m in month_list():
            if m in have or (time.time() - t0) / 60 > CWC_PROBE_BUDGET_MIN:
                continue
            days = pd.date_range(m.start_time, m.end_time, freq="W-THU")
            found = False
            for day in days:
                d = day.strftime("%d%m%Y")
                for p in CWC_PATTERNS:
                    url = CWC_BASE + p.format(d=d, dd=day.strftime("%d"), mm=day.strftime("%m"), yyyy=day.year)
                    try:
                        if requests.head(url, timeout=8, allow_redirects=True).status_code != 200:
                            continue
                        t = cwc_fetch(url)
                        res = cwc_parse_text(t, url) if t else []
                    except Exception:
                        continue
                    if res:
                        rows += res; status.append((url, "ok (probed)")); found = True; break
                if found:
                    break
    return pd.DataFrame(rows), status

def cwc_synthetic():
    # SMOKE TEST ONLY: storage that follows the synthetic water area, with noise.
    rng = np.random.default_rng(4); rows = []
    for n in RES_ORDER:
        for v in list(SC[n].values())[::3]:
            d = v["day"] + pd.Timedelta(days=int(rng.integers(-5, 6)))
            ph = (d.dayofyear - 290) / 365.25 * 2 * np.pi
            pct = float(np.clip(100 * (0.55 + 0.4 * np.cos(ph)) + rng.normal(0, 6), 2, 100))
            rows.append(dict(reservoir=n, date=d.normalize(), live_bcm=RESERVOIRS[n]["live_bcm"] * pct / 100,
                             capacity_bcm=RESERVOIRS[n]["live_bcm"], pct=pct, source="synthetic"))
    return pd.DataFrame(rows), [("synthetic", "ok")]

CWC_STATUS = "not run"
try:
    CWC, CWC_LOG = cwc_synthetic() if SMOKE_TEST else cwc_load_real()
    if len(CWC):
        CWC = CWC.drop_duplicates(["reservoir", "date"]).sort_values(["reservoir", "date"]).reset_index(drop=True)
        CWC_STATUS = "run"
    ok = sum(1 for _, s in CWC_LOG if s.startswith("ok"))
    print(f"CWC bulletins: {ok} parsed of {len(CWC_LOG)} tried · {len(CWC)} reservoir rows")
    bad = [u for u, s in CWC_LOG if not s.startswith("ok")]
    if bad:
        print("  not used (download or parse failed):", len(bad), "e.g.", bad[:3])
    if len(CWC):
        print(CWC.groupby("reservoir").agg(rows=("pct", "size"), first=("date", "min"), last=("date", "max"),
                                           capacity=("capacity_bcm", "median")).to_string())
except Exception as e:
    CWC, CWC_LOG, CWC_STATUS = pd.DataFrame(), [], f"failed ({type(e).__name__}: {str(e)[:80]})"
    print("CWC bulletins failed:", CWC_STATUS)

## 3 · Method: find each reservoir, count its water every month

1. **Water.** A pixel is water if MNDWI > `MNDWI_THR` (0) and SCL does not call it cloud or cloud shadow.
2. **The reservoir's zone.** Water frequency per pixel over all the clear looks in the run; pixels that are water in
   at least 5% of them, widened by 2 pixels, split into connected bodies; the zone is every body that comes within
   `SEED_RADIUS_KM` (5 km) of the dam point. Other ponds and rivers in the window are left out.
3. **Monthly area.** Water pixels inside the zone × 0.0036 km² (60 m × 60 m). A month is used only if clouds or
   missing data cover at most 5% of the zone.
4. **Fill index.** The month's area as a % of the largest area seen in the run (an estimate of "how full", by area).
5. **Dry-season drawdown.** For the headline cycle: the largest area in Sep–Dec of 2024 against the smallest in
   Mar–Jun of 2025.
6. **Check against CWC.** Each bulletin row is paired with the clearest image within 12 days. Spearman's rank
   correlation (the area–storage relation of a valley is curved, but it always rises together).

In [ ]:
# CELL: aggregate
from scipy import ndimage

def zone_for(n):
    sc = list(SC[n].values())
    if not sc:
        return None
    code = np.stack([v["code"] for v in sc])
    valid = (code > -127)
    water = valid & (code > MNDWI_THR * 100)
    freq = water.sum(0) / np.maximum(valid.sum(0), 1)
    cand = (freq >= ZONE_MIN_FREQ) & (valid.sum(0) >= 3)
    cand = ndimage.binary_dilation(cand, iterations=ZONE_DILATE_PX)
    lab, k = ndimage.label(cand)
    x, y = GRID[n]
    sx, sy = TO_UTM.transform(RESERVOIRS[n]["seed"][1], RESERVOIRS[n]["seed"][0])
    X, Y = np.meshgrid(x, y)
    near = np.hypot(X - sx, Y - sy) <= SEED_RADIUS_KM * 1000
    keep = np.unique(lab[near & (lab > 0)])
    z = np.isin(lab, keep) & (lab > 0)
    return dict(mask=z, freq=freq, edge=bool(z[0].any() or z[-1].any() or z[:, 0].any() or z[:, -1].any()))

ZONES = {n: zone_for(n) for n in RES_ORDER}
rows = []
for n in RES_ORDER:
    z = ZONES[n]
    if z is None:
        continue
    zm = z["mask"]; nz = int(zm.sum())
    for m, v in SC[n].items():
        c = v["code"][zm]
        bad = float(((c == -127) | (c == -128)).mean()) if nz else 1.0
        rows.append(dict(reservoir=n, month=pd.Period(m, "M"), day=v["day"], cloud_zone=bad,
                         area=(c > MNDWI_THR * 100).sum() * PIXEL_KM2,
                         area_lo=(c > MNDWI_ROBUST[0] * 100).sum() * PIXEL_KM2,
                         area_hi=(c > MNDWI_ROBUST[1] * 100).sum() * PIXEL_KM2,
                         area_scl=(v["sclw"][zm] & (c > -127)).sum() * PIXEL_KM2, zone_km2=nz * PIXEL_KM2))
AREA_ALL = pd.DataFrame(rows)
AREA = AREA_ALL[AREA_ALL.cloud_zone <= ZONE_CLOUD_MAX].copy()
AREA["fill"] = AREA.area / AREA.groupby("reservoir").area.transform("max")
for c in ("area_lo", "area_hi", "area_scl"):
    AREA["fill_" + c[5:]] = AREA[c] / AREA.groupby("reservoir")[c].transform("max")

def cycle(df, col, y):
    pk = df[(df.day.dt.year == y - 1) & df.day.dt.month.isin(PEAK_MONTHS)]
    lw = df[(df.day.dt.year == y) & df.day.dt.month.isin(LOW_MONTHS)]
    if pk.empty or lw.empty:
        return dict(peak=np.nan, low=np.nan, draw=np.nan, peak_day=None, low_day=None)
    p, l = pk.loc[pk[col].idxmax()], lw.loc[lw[col].idxmin()]
    return dict(peak=p[col], low=l[col], draw=l[col] / p[col] - 1 if p[col] > 0 else np.nan, peak_day=p.day, low_day=l.day)

def spearman(a, b):
    return float(pd.Series(a).rank().corr(pd.Series(b).rank())) if len(a) >= 3 else float("nan")

PAIRS = pd.DataFrame()
if CWC_STATUS == "run":
    pr = []
    for r in CWC.itertuples():
        a = AREA[AREA.reservoir == r.reservoir]
        if a.empty:
            continue
        dd = (a.day - r.date).abs()
        if dd.min() <= pd.Timedelta(days=MATCH_DAYS):
            s = a.loc[dd.idxmin()]
            pr.append(dict(reservoir=r.reservoir, date=r.date, day=s.day, pct=r.pct, area=s.area, fill=s.fill))
    PAIRS = pd.DataFrame(pr)

RES = {}
for n in RES_ORDER:
    a = AREA[AREA.reservoir == n].sort_values("day")
    p = PAIRS[PAIRS.reservoir == n] if len(PAIRS) else PAIRS
    RES[n] = dict(n_months=len(a), max_area=a.area.max() if len(a) else np.nan,
                  min_area=a.area.min() if len(a) else np.nan,
                  cyc=cycle(a, "area", END_YEAR),
                  cyc_lo=cycle(a, "area_lo", END_YEAR), cyc_hi=cycle(a, "area_hi", END_YEAR),
                  cyc_scl=cycle(a, "area_scl", END_YEAR),
                  cyc_prev=[cycle(a, "area", y) for y in range(int(DATA_START[:4]) + 1, END_YEAR)],
                  rho=spearman(p.area, p.pct) if len(p) else np.nan, n_pairs=len(p),
                  zone_km2=a.zone_km2.iloc[0] if len(a) else np.nan, edge=ZONES[n]["edge"] if ZONES[n] else None)
    r = RES[n]
    print(f"{n:15s} {r['n_months']:2d} clear months · area {r['min_area']:.0f}-{r['max_area']:.0f} km² · "
          f"drawdown {END_YEAR - 1}->{END_YEAR} {r['cyc']['draw']:+.0%} · CWC rho {r['rho']:.2f} ({r['n_pairs']} pairs)")

## 4 · Results: the headline, after the sanity checks

In [ ]:
# CELL: results
PROBLEMS, NOTES = [], []
def pct(v, d=0):
    return f"{v:+.{d}%}" if v == v else "n/a"

for n in RES_ORDER:
    r, cfg = RES[n], RESERVOIRS[n]
    if r["n_months"] < MIN_CLEAR_MONTHS:
        PROBLEMS.append(f"{n}: only {r['n_months']} clear months (< {MIN_CLEAR_MONTHS}): too few to describe the cycle")
    if cfg["area_km2"] and r["max_area"] == r["max_area"]:
        ratio = r["max_area"] / cfg["area_km2"]
        if not (AREA_RATIO_RANGE[0] <= ratio <= AREA_RATIO_RANGE[1]):
            PROBLEMS.append(f"{n}: largest water area {r['max_area']:.0f} km² is {ratio:.0%} of the documented "
                            f"{cfg['area_km2']:.0f} km²: window, zone or threshold is wrong")
    if r["edge"]:
        NOTES.append(f"{n}: the zone touches the edge of its window; the reservoir may continue outside it (area underestimated)")
    a = AREA[AREA.reservoir == n]
    for fd in cfg["full_dates"]:
        fd = pd.Timestamp(fd)
        near = a[(a.day >= fd - pd.Timedelta(days=10)) & (a.day <= fd + pd.Timedelta(days=40))]
        if near.empty:
            NOTES.append(f"{n}: no clear image near the documented full date {fd:%d %b %Y}: check skipped")
        elif near.fill.max() < FULL_DATE_MIN_FILL:
            PROBLEMS.append(f"{n}: documented full on {fd:%d %b %Y}, but the fill index nearby is only {near.fill.max():.0%}")
    pk = a[a.day.dt.month.isin([10, 11, 12])].area.mean(); lw = a[a.day.dt.month.isin([4, 5, 6])].area.mean()
    if pk == pk and lw == lw and pk <= lw:
        PROBLEMS.append(f"{n}: Apr-Jun area ({lw:.0f} km²) not below Oct-Dec ({pk:.0f} km²): contradicts the monsoon fill cycle")
    d = r["cyc"]["draw"]
    if d == d:
        if d > 0:
            PROBLEMS.append(f"{n}: water area grew {d:+.0%} through the dry season: contradicts the known drawdown")
        if abs(d) > MAX_DRAWDOWN:
            PROBLEMS.append(f"{n}: drawdown {d:+.0%} beyond -{MAX_DRAWDOWN:.0%}: implausible for a major reservoir")
        for k, lab in (("cyc_lo", f"MNDWI > {MNDWI_ROBUST[0]}"), ("cyc_hi", f"MNDWI > {MNDWI_ROBUST[1]}"), ("cyc_scl", "SCL water class")):
            v = r[k]["draw"]
            if v == v and abs(v - d) > 0.15:
                NOTES.append(f"robustness: {n} drawdown {d:+.0%} becomes {v:+.0%} with {lab}: quote it as a range")
    else:
        PROBLEMS.append(f"{n}: no clear image in Sep-Dec {END_YEAR - 1} or Mar-Jun {END_YEAR}: no headline cycle")
    if len(a) >= 5:
        dis = ((a.area - a.area_scl).abs() / a.area.where(a.area > 0)).median()
        if dis > SCL_AGREE_MAX:
            PROBLEMS.append(f"{n}: MNDWI and SCL water areas differ by a median {dis:.0%}: check the offset/harmonisation")
    if a.fill.max() > 1.0001:
        PROBLEMS.append(f"{n}: fill index above 100%: arithmetic error")
    if CWC_STATUS == "run":
        if r["n_pairs"] >= MIN_CWC_PAIRS:
            if r["rho"] < 0:
                PROBLEMS.append(f"{n}: satellite area and CWC storage move in opposite directions (rho = {r['rho']:.2f})")
            elif r["rho"] < MIN_RHO:
                NOTES.append(f"{n}: weak agreement with CWC (rho = {r['rho']:.2f}): say so in the post")
        else:
            NOTES.append(f"{n}: only {r['n_pairs']} bulletin dates within {MATCH_DAYS} days of a clear image: comparison is thin")
if CWC_STATUS != "run":
    NOTES.append(f"CWC comparison {CWC_STATUS}: the post must not claim a check against published storage")
# processing baseline: every scene after the cutoff must have had the offset removed
if len(BASELINES):
    late = BASELINES[[pd.Timestamp(v["day"]) >= pd.Timestamp(BASELINE_CUTOFF) for n in RES_ORDER for v in SC[n].values()]]
    if len(late) and not late.offset.all():
        PROBLEMS.append("some scenes after 25 Jan 2022 were read without removing the +1000 offset: MNDWI is biased")
if SMOKE_TEST:
    PROBLEMS.append("SMOKE TEST: synthetic images and bulletins")

print("Assumptions:", dict(RES_M=RES_M, MNDWI_THR=MNDWI_THR, ZONE_CLOUD_MAX=ZONE_CLOUD_MAX, SEED_RADIUS_KM=SEED_RADIUS_KM,
      ZONE_MIN_FREQ=ZONE_MIN_FREQ, MATCH_DAYS=MATCH_DAYS, period=(DATA_START, DATA_END)))
for n_ in NOTES:
    print("note:", n_)
if PROBLEMS:
    print("\n" + "!" * 78)
    for p in PROBLEMS:
        print("!! SANITY CHECK FAILED:", p)
    print("!! Do NOT post these numbers until every failure above is explained or fixed.")
    print("!" * 78 + "\n")
else:
    print("All sanity checks passed.")

DRAW = {n: RES[n]["cyc"]["draw"] for n in RES_ORDER}
TOP = max(RES_ORDER, key=lambda n: abs(DRAW[n]) if DRAW[n] == DRAW[n] else -1)
RHOS = [RES[n]["rho"] for n in RES_ORDER if RES[n]["n_pairs"] >= MIN_CWC_PAIRS]
RHO_MED = float(np.median(RHOS)) if RHOS else float("nan")
HEADLINE = (f"From the {END_YEAR - 1} post-monsoon peak to the {END_YEAR} pre-monsoon low, the open-water area measured "
            "by Sentinel-2 changed by " + ", ".join(f"{pct(DRAW[n])} at {n}" for n in RES_ORDER) + " (estimate).")
CWC_LINE = (f"Satellite area moves with CWC's published storage: median rank correlation {RHO_MED:.2f} over "
            f"{len(RHOS)} reservoirs ({len(PAIRS)} bulletin dates)." if RHOS else
            "Not compared with CWC storage in this run (too few bulletin dates matched).")
BIG = pct(DRAW[TOP])
c = RES[TOP]["cyc"]
BIG_LINE = (f"Water surface at {TOP}, {c['peak_day']:%b %Y} to {c['low_day']:%b %Y}: {c['peak']:.0f} → {c['low']:.0f} km²."
            if c["peak_day"] is not None else f"Water surface at {TOP}.")
print("\nHEADLINE (estimate):", HEADLINE)
print("  ", CWC_LINE)
print("\nRobustness (drawdown of the headline cycle):")
for n in RES_ORDER:
    r = RES[n]
    prev = " · ".join(f"{(x['low_day'] or pd.Timestamp('1900')).year}: {pct(x['draw'])}" for x in r["cyc_prev"])
    print(f"  {n:15s} {pct(r['cyc']['draw'])} · MNDWI>{MNDWI_ROBUST[0]} {pct(r['cyc_lo']['draw'])} · MNDWI>{MNDWI_ROBUST[1]} "
          f"{pct(r['cyc_hi']['draw'])} · SCL {pct(r['cyc_scl']['draw'])} · earlier cycles {prev}")

In [ ]:
# CELL: results_chart
COL = dict(zip(RES_ORDER, [B.MOSS, B.AMBER, B.INK, B.STONE]))
TAG = " · SMOKE TEST, NOT REAL" if SMOKE_TEST else ""

def draw_series(ax, n, upto=None, big=False, cwc=True):
    a = AREA[AREA.reservoir == n].sort_values("day")
    if upto is not None:
        a = a[a.day <= upto]
    for y in range(int(DATA_START[:4]), int(DATA_END[:4]) + 1):
        ax.axvspan(pd.Timestamp(y, MONSOON_MONTHS[0], 1), pd.Timestamp(y, MONSOON_MONTHS[-1], 30), color=B.PAPER_DEEP, lw=0, zorder=0)
    ax.plot(a.day, a.fill * 100, color=B.MOSS, lw=2.6 if big else 1.8, marker="o", ms=4 if big else 3,
            label="satellite: % of largest water area")
    if cwc and CWC_STATUS == "run":
        c = CWC[CWC.reservoir == n]
        if upto is not None:
            c = c[c.date <= upto]
        ax.scatter(c.date, c.pct, color=B.AMBER, s=40 if big else 22, zorder=3, label="CWC: % of live capacity")
    ax.set_xlim(pd.Timestamp(DATA_START), pd.Timestamp(DATA_END)); ax.set_ylim(0, 108)
    ax.xaxis.set_major_formatter(matplotlib.dates.DateFormatter("%Y")); ax.xaxis.set_major_locator(matplotlib.dates.YearLocator())
    ax.tick_params(labelsize=13 if big else 9)

def class_map(n, month):
    v = SC[n][month]; c = v["code"]
    img = np.zeros(c.shape, np.uint8)                              # 0 land, 1 water, 2 cloud / no data
    img[c > MNDWI_THR * 100] = 1; img[c <= -127] = 2
    return img

MAP_CMAP = matplotlib.colors.ListedColormap([B.PAPER_DEEP, B.MOSS, B.STONE])
def draw_map(ax, n, month, big=False):
    x, y = GRID[n]
    ext = [0, (x[-1] - x[0]) / 1000, 0, (y[0] - y[-1]) / 1000]
    ax.imshow(class_map(n, month), cmap=MAP_CMAP, vmin=0, vmax=2, interpolation="nearest", extent=ext)
    if ZONES[n] is not None:
        ax.contour(ZONES[n]["mask"][::-1].astype(float), levels=[0.5], colors=[B.AMBER], linewidths=1.6 if big else 1,
                   extent=ext)
    ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
    for s in ax.spines.values():
        s.set_visible(False)

fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), sharex=True)
for ax, n in zip(axes.flat, RES_ORDER):
    draw_series(ax, n)
    ax.text(0.01, 0.97, n.upper(), transform=ax.transAxes, fontproperties=B.FP.get("mono_bold"), fontsize=11, color=B.INK, va="top")
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc="upper right", ncol=2, fontsize=10)
B.chart_title(axes[0, 0], "How full, month by month", f"Sentinel-2 water area vs CWC storage · shaded: monsoon · {RES_M} m grid")
B.source_note(fig, f"23N Field Test #{FT_NUM} · {DATA_CREDIT}{TAG}")
fig.tight_layout(rect=(0, 0, 1, 0.95)); fig.savefig(f"ft{FT_NUM}_monthly.png", dpi=150); plt.show()

if len(PAIRS):
    fig, ax = plt.subplots(figsize=(7, 6))
    for n in RES_ORDER:
        p = PAIRS[PAIRS.reservoir == n]
        ax.scatter(p.pct, p.fill * 100, s=30, color=COL[n], label=f"{n} (ρ {RES[n]['rho']:.2f})")
    ax.plot([0, 100], [0, 100], color=B.LINE, lw=1)
    ax.set_xlabel("CWC live storage, % of capacity"); ax.set_ylabel("satellite water area, % of largest")
    ax.legend(fontsize=9)
    B.chart_title(ax, "Satellite vs published storage", f"bulletin dates within {MATCH_DAYS} days of a clear image")
    B.source_note(fig, f"23N Field Test #{FT_NUM} · {DATA_CREDIT}{TAG}")
    fig.tight_layout(); fig.savefig(f"ft{FT_NUM}_cwc.png", dpi=150); plt.show()

c = RES[TOP]["cyc"]
if c["peak_day"] is not None:
    x, y = GRID[TOP]; asp = (x[-1] - x[0]) / max(y[0] - y[-1], 1)
    if asp > 1.3:                                                   # wide window: stack the two maps
        fig, axes = plt.subplots(2, 1, figsize=(11, min(2 * 11 / asp + 1.6, 12)))
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, min(6 / asp + 1.4, 9)))
    for ax, d in zip(axes, (c["peak_day"], c["low_day"])):
        draw_map(ax, TOP, str(d.to_period("M")))
        ax.text(0.02, 0.96, f"{d:%d %b %Y}", transform=ax.transAxes, fontproperties=B.FP.get("mono_bold"), fontsize=13,
                color=B.INK, va="top", bbox=dict(boxstyle="round,pad=0.25", fc=B.PAPER, ec="none"))
    B.chart_title(axes[0], f"{TOP}: peak and low", "moss: water (MNDWI > 0) · grey: cloud · amber: reservoir zone")
    B.source_note(fig, f"23N Field Test #{FT_NUM} · {DATA_CREDIT}{TAG}")
    fig.tight_layout(); fig.savefig(f"ft{FT_NUM}_maps.png", dpi=150); plt.show()

## 5 · Interactive demo
Pick a reservoir and a month: you see that month's water map with the reservoir zone, and the fill-index series.
The cell doesn't block "Run all"; open the public `gradio.live` link it prints.

In [ ]:
# CELL: gradio
def demo_figure(n, month):
    fig = plt.figure(figsize=(10, 9))
    ax1 = fig.add_axes([0.03, 0.40, 0.94, 0.55]); ax2 = fig.add_axes([0.08, 0.06, 0.88, 0.27])
    if month in SC[n]:
        draw_map(ax1, n, month); ax1.set_title(f"{n}, {SC[n][month]['day']:%d %b %Y}", loc="left", fontsize=14)
    draw_series(ax2, n)
    a = AREA[(AREA.reservoir == n) & (AREA.month.astype(str) == month)]
    txt = (f"{a.area.iloc[0]:.0f} km² of open water · {a.fill.iloc[0]:.0%} of the largest area seen" if len(a)
           else "no clear image this month (cloud over the reservoir, or not read)")
    B.source_note(fig, f"{DATA_CREDIT}{TAG}")
    return fig, txt

try:
    import gradio as gr
    months = [str(m) for m in month_list()]
    with gr.Blocks(title=f"23N Field Test #{FT_NUM}: reservoirs from space") as demo:
        gr.Markdown(f"### 23N Field Test #{FT_NUM}: Gujarat's reservoirs from Sentinel-2\n"
                    "Independent study on public data. Water area is an estimate, not storage volume.")
        dd_r = gr.Dropdown(RES_ORDER, value=RES_ORDER[0], label="Reservoir")
        dd_m = gr.Dropdown(months, value=months[-3], label="Month")
        out_plot, out_txt = gr.Plot(), gr.Textbox(label="This month")
        for w in (dd_r, dd_m):
            w.change(demo_figure, [dd_r, dd_m], [out_plot, out_txt])
    demo.launch(share=True, prevent_thread_lock=True)
except Exception as e:
    print("Gradio demo not started:", type(e).__name__, e)

## 6 · Video (1080×1080, 30 fps, 58 s timeline saved at 1.5×, H.264) and cover (1080×1350)
Every map, line and number in the video comes from the run above.

In [ ]:
# CELL: video_lib
VIDEO_PATH, COVER_PATH = f"ft{FT_NUM}_video.mp4", f"ft{FT_NUM}_cover.png"
FPS = 30
def _ease(p): p = min(max(p, 0.0), 1.0); return 1 - (1 - p) ** 3
def _prog(t, a, b): return min(max((t - a) / (b - a), 0.0), 1.0)
def _title(fig, title, sub):
    fig.text(0.055, 0.85, title, fontproperties=B.FP.get("display"), fontsize=40, color=B.INK, va="top")
    fig.text(0.055, 0.785, sub.upper(), fontproperties=B.FP.get("mono"), fontsize=14, color=B.MUTED, va="top")

TOP_MONTHS = sorted(AREA[AREA.reservoir == TOP].month.astype(str))
MAP_SECONDS, SERIES_END = 14.0, 27.0
def map_index(t):
    return min(int(t / (MAP_SECONDS / max(len(TOP_MONTHS), 1))), max(len(TOP_MONTHS) - 1, 0))

def draw_evidence(fig, t):
    # t runs 0..35 inside the evidence section (video seconds 5..40)
    B.header(fig, FT_NUM)
    if t < MAP_SECONDS and TOP_MONTHS:
        m = TOP_MONTHS[map_index(t)]
        a = AREA[(AREA.reservoir == TOP) & (AREA.month.astype(str) == m)].iloc[0]
        _title(fig, f"{TOP} from space", f"Sentinel-2 · water (MNDWI > 0) · {RES_M} m grid")
        x, y = GRID[TOP]; asp = (y[0] - y[-1]) / max(x[-1] - x[0], 1)
        w = 0.89; h = min(0.56, w * asp * 1.0)
        ax = fig.add_axes([0.055, 0.46 - h / 2, w, h]); draw_map(ax, TOP, m, big=True)
        fig.text(0.945, 0.785, f"{a.day:%b %Y}", fontproperties=B.FP.get("mono_bold"), fontsize=22, color=B.MOSS, ha="right", va="top")
        fig.text(0.055, 0.205, f"{a.area:.0f} km²", fontproperties=B.FP.get("display"), fontsize=44, color=B.MOSS, va="top")
        fig.text(0.33, 0.19, f"{a.fill:.0%} OF THE LARGEST AREA SEEN", fontproperties=B.FP.get("mono"), fontsize=15, color=B.MUTED, va="center")
        B.caption(fig, "Moss: open water. Grey: cloud. Amber line: the reservoir's zone")
    elif t < SERIES_END:
        _title(fig, "Four reservoirs, four years", "satellite fill index (line) · CWC storage (dots) · shaded: monsoon")
        span = pd.Timestamp(DATA_END) - pd.Timestamp(DATA_START)
        upto = pd.Timestamp(DATA_START) + span * _ease(_prog(t, MAP_SECONDS + 0.3, SERIES_END - 2))
        for i, n in enumerate(RES_ORDER):
            ax = fig.add_axes([0.08 + (i % 2) * 0.46, 0.47 - (i // 2) * 0.26, 0.40, 0.2])
            draw_series(ax, n, upto=upto, big=False); ax.tick_params(labelsize=11)
            ax.text(0.02, 0.95, n.upper(), transform=ax.transAxes, fontproperties=B.FP.get("mono_bold"), fontsize=12, color=B.INK, va="top")
        B.caption(fig, "Each monsoon refills them; the dry season draws them down")
    else:
        _title(fig, f"Dry season {END_YEAR}", f"change in water area · {END_YEAR - 1} peak to {END_YEAR} low")
        ax = fig.add_axes([0.13, 0.22, 0.82, 0.50])
        g = _ease(_prog(t, SERIES_END + 0.1, SERIES_END + 2.5))
        vals = [DRAW[n] * 100 * g if DRAW[n] == DRAW[n] else 0 for n in RES_ORDER]
        ax.bar(range(len(RES_ORDER)), vals, color=[B.MOSS if n == TOP else B.STONE for n in RES_ORDER], width=0.6)
        ax.axhline(0, color=B.INK, lw=1)
        lim = max([10] + [abs(DRAW[n]) * 100 * 1.3 for n in RES_ORDER if DRAW[n] == DRAW[n]])
        ax.set_ylim(-lim, max(lim * 0.15, 5)); ax.set_xticks(range(len(RES_ORDER)), RES_ORDER, fontsize=15)
        ax.tick_params(axis="y", labelsize=13); ax.set_ylabel("% change in water area", fontsize=14)
        if g >= 1:
            for i, n in enumerate(RES_ORDER):
                if DRAW[n] == DRAW[n]:
                    ax.text(i, DRAW[n] * 100 - lim * 0.05, pct(DRAW[n]), ha="center", va="top",
                            fontproperties=B.FP.get("mono_bold"), fontsize=16, color=B.INK)
        B.caption(fig, CWC_LINE)

def render_video(path=VIDEO_PATH):
    v = B.Video(path, fps=FPS)
    B.hook_card(v.fig, FT_NUM, "How full are Gujarat's reservoirs this month?",
                f"Sentinel-2 water area · Sardar Sarovar, Ukai, Kadana, Dharoi · {DATA_START[:4]}-{DATA_END[:4]}"); v.hold(5)
    n = int(35 * FPS); key0, buf = None, None
    for f in range(n):
        t = f / FPS
        if t < MAP_SECONDS:   key = ("m", map_index(t))
        elif t < SERIES_END:  key = ("s", min(round(t * 8) / 8, SERIES_END - 2))
        else:                 key = ("b", min(round(t * 10) / 10, SERIES_END + 2.6))
        if key != key0:
            v.clear(); draw_evidence(v.fig, t); buf = v.grab(); key0 = key
        v.proc.stdin.write(buf); v.frames += 1
    v.clear(); B.headline_card(v.fig, FT_NUM, BIG, BIG_LINE,
                               f"Sentinel-2 L2A, MNDWI > {MNDWI_THR}, SCL cloud mask, {RES_M} m, clearest date per month, "
                               f"checked against CWC bulletins. Area, not volume. Estimate.")
    v.hold(12)
    v.clear(); B.end_card(v.fig, FT_NUM, DATA_CREDIT); v.hold(6)
    v.close()
    print("video frames:", v.frames, f"({v.frames / FPS:.1f}s timeline)")

def render_cover(path=COVER_PATH):
    c = RES[TOP]["cyc"]
    def panel(ax):
        if c["peak_day"] is None:
            return
        ax.set_axis_off()
        a1 = ax.inset_axes([0, 0.52, 1, 0.46]); a2 = ax.inset_axes([0, 0.0, 1, 0.46])
        for a, d in ((a1, c["peak_day"]), (a2, c["low_day"])):
            draw_map(a, TOP, str(d.to_period("M")), big=True)
            a.text(0.015, 0.95, f"{d:%b %Y} · {AREA[(AREA.reservoir == TOP) & (AREA.day == d)].area.iloc[0]:.0f} km²",
                   transform=a.transAxes, fontproperties=B.FP.get("mono_bold"), fontsize=14, color=B.INK, va="top",
                   bbox=dict(boxstyle="round,pad=0.25", fc=B.PAPER, ec="none"))
    B.cover(path, FT_NUM, "Gujarat's reservoirs, measured from space", f"{TOP} · Sentinel-2 water area · {END_YEAR - 1}-{END_YEAR}",
            BIG, f"water area at {TOP}, {END_YEAR - 1} peak to {END_YEAR} low (estimate)", image=panel,
            data_credit="Copernicus Sentinel data · CWC")

In [ ]:
# CELL: video_render
render_video()
render_cover()
_probe = subprocess.run(["ffprobe", "-v", "error", "-select_streams", "v:0", "-show_entries",
                         "stream=width,height,r_frame_rate,codec_name:format=duration", "-of", "json", VIDEO_PATH],
                        capture_output=True, text=True)
print(_probe.stdout)
try:
    from google.colab import files
    files.download(VIDEO_PATH); files.download(COVER_PATH)
except Exception:
    print("Not on Colab: video and cover saved in the working directory.")

## 7 · Post text (ready to paste once the sanity checks pass)

In [ ]:
# CELL: post_text
POST_LINKEDIN_TEMPLATE = """How full are Gujarat's reservoirs? We measured the water surface of Sardar Sarovar, Ukai, Kadana and Dharoi from space, month by month, {period}.

23N Field Test #10: {headline}

{cwc_fact}, published as weekly PDF tables of gauge readings. A satellite can check them independently.

How we tested it:
• Copernicus Sentinel-2, {n_images} clear monthly images at {res_m} m; water mapped with MNDWI and a cloud mask
• Each reservoir's zone found from the images; cloudy months dropped
• {cwc_line}

Result: {result_line}

Limits: water area is not storage volume; clouds hide most monsoon months; {res_m} m pixels miss narrow arms. An estimate.

Notebook in the comments.

{signature}
"""
POST_X_TEMPLATE = """Water area of 4 Gujarat reservoirs from Sentinel-2, {y0} peak to {y1} low: {x_changes}. {cwc_short} 23N Field Test #10. Notebook below.

---
Thread (optional):
1/ Data: Copernicus Sentinel-2 L2A via Microsoft Planetary Computer, no login. {res_m} m grid, clearest date each month.
2/ Method: MNDWI > 0 inside each reservoir's own zone, cloud-masked; checked against CWC's weekly storage bulletins.
3/ Limits: area, not volume; monsoon clouds hide the refill. Independent study on public data.
"""
def _load_template(name, fallback):
    try:
        return open(name, encoding="utf-8").read()
    except Exception:
        return fallback
RESULT_LINE = "water area, peak → low: " + "; ".join(f"{n} {RES[n]['cyc']['peak']:.0f} → {RES[n]['cyc']['low']:.0f} km²" for n in RES_ORDER) + "."
POST_VALUES = dict(headline=HEADLINE, cwc_fact=CWC_FACT, n_images=f"{len(AREA):,}", period=f"{DATA_START[:4]}–{DATA_END[:4]}",
                   res_m=RES_M, cwc_line=CWC_LINE, result_line=RESULT_LINE, y0=END_YEAR - 1, y1=END_YEAR,
                   x_changes=", ".join(f"{n} {pct(DRAW[n])}" for n in RES_ORDER),
                   cwc_short=(f"Median rank correlation with CWC storage: {RHO_MED:.2f}." if RHOS else "Not compared with CWC storage."),
                   signature=B.post_signature(FT_NUM) + " #Water #EarthObservation #Gujarat")
post_li = _load_template("post_linkedin.md", POST_LINKEDIN_TEMPLATE).format(**POST_VALUES)
post_x = _load_template("post_x.md", POST_X_TEMPLATE).format(**POST_VALUES)
x_main = post_x.split("---")[0].strip()
if PROBLEMS:
    print("!! SANITY CHECKS FAILED (see Results). Do NOT post this text.\n")
print("=" * 30, f"LinkedIn ({len(post_li)} chars)", "=" * 30); print(post_li)
print("=" * 30, f"X (main post {len(x_main)} chars)", "=" * 30); print(post_x)
if len(post_li) > 1300: print(f"WARNING: LinkedIn post is {len(post_li)} chars (> 1,300): trim before posting")
if len(x_main) > 280: print(f"WARNING: X post is {len(x_main)} chars (> 280): trim before posting")

## 8 · Limitations (read before quoting the number)

- **Area, not volume.** A reservoir in a deep gorge (Sardar Sarovar) can lose a lot of storage while its surface
  shrinks little; a shallow one does the opposite. The fill index is a % of the largest *area* seen, not a % of live
  capacity. The CWC comparison uses rank correlation for this reason.
- **Clouds.** Sentinel-2 can't see through cloud. Most monsoon months are dropped, so the rise is seen late and the
  peak may be missed by weeks. Radar (Sentinel-1) would fill this gap.
- **One image a month, 60 m pixels.** Narrow arms and shallow edges smaller than a pixel are missed or mixed;
  water under floating vegetation or in deep shadow can be missed; dark, wet soil can be counted.
- **The windows.** Each reservoir is searched for inside a fixed lon/lat window. Sardar Sarovar's backwater runs
  214 km upstream; if its zone touches the window edge the notebook says so and its area is an underestimate.
- **The zone is learnt from the run.** Pixels that are water in at least 5% of clear looks; a new shoreline at a
  record level would be partly outside it.
- **CWC bulletins.** Parsed from PDF text; a row is kept only if its live capacity matches the known value. Bulletin
  file names vary, so some weeks are missing; a bulletin and an image can be up to 12 days apart.
- **Not a statement about any dam authority, water board or ministry.** The reservoirs are used only as places.
- **Licences.** Copernicus Sentinel data terms: free, full and open; attribution "Contains modified Copernicus
  Sentinel data 2022–2025". Access via Microsoft Planetary Computer (no account). CWC bulletins: public Government
  of India documents, cited as the source of the storage figures. Code in this repo: see LICENSE.